# Chapter 4: Choosing speech recognition for your callers

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [Choosing speech recognition for your callers](https://books.sugiv.fyi/voice_applications/choosing-asr.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## Word error rate, from scratch

In [ ]:
import json
import re
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np


def edit_ops(ref, hyp):
    """Minimum (substitutions, deletions, insertions) turning ref into hyp."""
    R, H = len(ref), len(hyp)
    d = np.zeros((R + 1, H + 1), dtype=int)
    d[:, 0], d[0, :] = np.arange(R + 1), np.arange(H + 1)
    for i in range(1, R + 1):
        for j in range(1, H + 1):
            d[i, j] = min(d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]), d[i - 1, j] + 1, d[i, j - 1] + 1)
    i, j, ops = R, H, {"sub": 0, "del": 0, "ins": 0}
    while i or j:  # walk back to count each kind of error
        if i and j and d[i, j] == d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops["sub"] += ref[i - 1] != hyp[j - 1]
            i, j = i - 1, j - 1
        elif i and d[i, j] == d[i - 1, j] + 1:
            ops["del"] += 1
            i -= 1
        else:
            ops["ins"] += 1
            j -= 1
    return ops


def wer(ref, hyp):
    ref, hyp = ref.split(), hyp.split()
    ops = edit_ops(ref, hyp)
    return sum(ops.values()) / max(len(ref), 1), ops


ref = "grace adeyemi about her canyon grizl"
hyp = "grace a daymi about her canyon grizzly"
print(wer(ref, hyp))

## Normalize before you score

In [ ]:
cap = json.load(open("data/ch3/capture_ch3.json"))
truth = "Hi, is my bike ready? The ticket number is HW-1042."
for model, res in cap["results"].items():
    hyp = res["c01_status/phone"]["text"]
    print(f"{model:30} WER {wer(truth.lower(), hyp.lower())[0]:5.0%}   {hyp}")

In [ ]:
DIGITS = "zero one two three four five six seven eight nine".split()
SMALL = {w: i for i, w in enumerate(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
    "fifteen sixteen seventeen eighteen nineteen".split())}
SMALL["oh"] = 0
TENS = {w: 10 * i for i, w in enumerate("_ _ twenty thirty forty fifty sixty seventy eighty ninety".split()) if w != "_"}


STOP = set("a i is it on to my me of in at be do so up or an as by if no we he us am hi ok".split())


def normalize(text):
    """One canonical form for scoring: 'HW-1042' and 'h w one oh four two' both become 'h w 1 0 4 2'."""
    t = text.lower().replace("-", " ").replace("'", "")
    toks = re.sub(r"[^a-z0-9 ]", " ", t).split()
    out, i = [], 0
    while i < len(toks):
        tok = toks[i]
        nxt = toks[i + 1] if i + 1 < len(toks) else ""
        if tok in TENS and nxt in SMALL and 0 < SMALL[nxt] < 10:
            out += list(str(TENS[tok] + SMALL[nxt]))  # twenty one -> 2 1
            i += 2
            continue
        if tok.isdigit() or re.fullmatch(r"[a-z]+\d+|\d+[a-z]+", tok):
            out += list(tok)  # 1042 -> 1 0 4 2, hw1042 -> h w 1 0 4 2, sl5 -> s l 5
        elif tok in SMALL:
            out += list(str(SMALL[tok]))  # one oh four two -> 1 0 4 2, ten -> 1 0
        elif tok == "o" and ((out and out[-1].isdigit()) or nxt in SMALL or nxt.isdigit()):
            out.append("0")  # 'seven o' sounds exactly like 'seven oh'
        elif tok in TENS:
            out += list(str(TENS[tok]))
        elif len(tok) <= 2 and tok not in STOP:
            out += list(tok)  # 'hw' as one token or as two letters scores the same
        else:
            out.append(tok)
        i += 1
    return " ".join(out)


for model, res in cap["results"].items():
    hyp = res["c01_status/phone"]["text"]
    print(f"{model:30} WER {wer(normalize(truth), normalize(hyp))[0]:5.0%}   {normalize(hyp)}")

## Score the words the business depends on

In [ ]:
calls = [json.loads(l) for l in open("data/calls/manifest.jsonl")]


def entity_hits(call, hyp):
    h = f" {normalize(hyp)} "
    return [(e["type"], f" {normalize(e['written'])} " in h) for e in call["entities"]]


by_id = {c["id"]: c for c in calls}
for model, res in cap["results"].items():
    hits = [hit for key, r in res.items() for hit in entity_hits(by_id[key.split("/")[0]], r["text"])]
    detail = ", ".join(f"{t} {'yes' if ok else 'NO'}" for t, ok in hits)
    print(f"{model:30} {sum(ok for _, ok in hits)}/{len(hits)} entities   {detail}")

## The test bench

In [ ]:
import pandas as pd

cap4 = json.load(open("data/ch4/capture_ch4.json"))
res4 = cap4["results"]
refs = {}
for root, name in [("data/calls", "halfwheel"), ("data/librispeech", "librispeech")]:
    for line in open(f"{root}/manifest.jsonl"):
        row = json.loads(line)
        refs[(name, row["id"])] = row


def score(model, fix=normalize, sets=("halfwheel", "librispeech")):
    """WER per (set, condition), entities heard on phone audio, speed and memory."""
    errs, words = defaultdict(int), defaultdict(int)
    ent = defaultdict(lambda: [0, 0])
    spent = audio = 0.0
    for key, r in res4[model]["files"].items():
        name, cond, uid = key.split("/")
        if name not in sets:
            continue
        ref, hyp = normalize(refs[(name, uid)]["written"]).split(), fix(r["text"]).split()
        errs[name, cond] += sum(edit_ops(ref, hyp).values())
        words[name, cond] += len(ref)
        spent, audio = spent + r["s"], audio + r["audio_s"]
        if name == "halfwheel" and cond == "phone":
            padded = f" {fix(r['text'])} "
            for e in refs[(name, uid)]["entities"]:
                ent[e["type"]][0] += f" {normalize(e['written'])} " in padded
                ent[e["type"]][1] += 1
    out = {f"{n} {c}": errs[n, c] / words[n, c] for n, c in words}
    out.update({t: hit / n for t, (hit, n) in ent.items()})
    out["entities"] = sum(h for h, _ in ent.values()) / sum(n for _, n in ent.values())
    out["x real time"] = audio / spent
    out["peak GB"] = res4[model]["peak_gb"]
    return out


BASE = {"parakeet-ctc-0.6b": "CTC", "parakeet-tdt-0.6b-v2": "TDT", "qwen3-asr-0.6b": "LLM",
        "qwen3-asr-1.7b": "LLM", "whisper-large-v3-turbo": "LLM", "kyutai-stt-2.6b-en": "streaming LM"}
table = pd.DataFrame({m: score(m) for m in BASE if "files" in res4.get(m, {})}).T
table.insert(0, "decoder", [BASE[m] for m in table.index])
pct = ["halfwheel clean", "halfwheel phone", "librispeech clean", "librispeech phone",
       "ticket", "phone", "bike", "name", "entities"]
(table[["decoder"] + pct + ["x real time", "peak GB"]]
 .style.format({c: "{:.1%}" for c in pct} | {"x real time": "{:.0f}x", "peak GB": "{:.1f}"}))

## Teaching the model the shop's words

In [ ]:
HINTS = ["parakeet-ctc-0.6b", "parakeet-ctc-0.6b + shop words (0.5)", "parakeet-ctc-0.6b + shop words (2.0)",
         "parakeet-tdt-0.6b-v2", "parakeet-tdt-0.6b-v2 + shop words (0.5)", "parakeet-tdt-0.6b-v2 + shop words (2.0)",
         "whisper-large-v3-turbo", "whisper-large-v3-turbo + shop words"]
hints = pd.DataFrame({m: score(m) for m in HINTS}).T
cols = ["halfwheel phone", "librispeech phone", "bike", "name", "phone", "ticket", "entities"]
hints[cols].style.format("{:.1%}")

In [ ]:
key = "halfwheel/phone/c04_message"
print("said:     ", refs[("halfwheel", "c04_message")]["written"])
for m in ["parakeet-ctc-0.6b", "parakeet-ctc-0.6b + shop words (2.0)"]:
    print(f"{m:38}", res4[m]["files"][key]["text"])

## Heard right, spelled wrong: snap to the catalogue

In [ ]:
import difflib
import functools

CATALOGUE = [  # what the shop knows before the phone rings
    "Trek Domane SL 5", "Specialized Tarmac SL7", "Cannondale Synapse", "Brompton C Line",
    "Giant Defy Advanced 2", "Canyon Grizl", "Surly Cross-Check", "Bianchi Oltre", "Orbea Orca",
    "Ribble Endurance SL", "Daniel Okafor", "Siobhan Murphy", "Tomasz Wojcik", "Mei Nguyen",
    "Raj Patel", "Grace Adeyemi", "Liam O'Sullivan", "Sofia Rossi", "Hannah Schmidt",
    "Kwame Mensah", "Aoife Byrne", "Diego Alvarez"]
CAT = [tuple(normalize(c).split()) for c in CATALOGUE]


@functools.lru_cache(maxsize=None)
def snap(text, cutoff=0.8):
    """Replace stretches of the transcript that closely match a catalogue entry with that entry."""
    words = normalize(text).split()
    candidates = []
    for i in range(len(words)):
        for entry in CAT:
            for n in {len(entry) - 1, len(entry), len(entry) + 1}:  # allow one word split or merged
                if n >= 1 and i + n <= len(words):
                    r = difflib.SequenceMatcher(None, " ".join(words[i:i + n]), " ".join(entry)).ratio()
                    if r >= cutoff:
                        candidates.append((r, i, n, entry))
    owner = [None] * len(words)  # best matches first; never let two matches overlap
    for r, i, n, entry in sorted(candidates, reverse=True):
        if all(o is None for o in owner[i:i + n]):
            owner[i:i + n] = [(i, entry)] * n
    out, k = [], 0
    while k < len(words):
        if owner[k] is None:
            out.append(words[k])
            k += 1
        else:
            start, entry = owner[k]
            out += entry
            while k < len(words) and owner[k] is not None and owner[k][0] == start:
                k += 1
    return " ".join(out)


print(snap("How much would a wheel true cost on a canyon grizzle?"))
print(snap("It's about the Brompton Sea Line."))
print(snap("Hi, it's Sophia Rossi. Is my bike ready?"))

In [ ]:
def false_snaps(model):
    """Catalogue entries that appear after snapping but were never said."""
    count = 0
    for key, r in res4[model]["files"].items():
        name, cond, uid = key.split("/")
        if name == "halfwheel" and cond == "phone":
            said, heard = f" {normalize(refs[(name, uid)]['written'])} ", f" {snap(r['text'])} "
            count += sum(f" {' '.join(e)} " in heard and f" {' '.join(e)} " not in said for e in CAT)
    return count


rows = {}
for m in list(BASE) + ["parakeet-tdt-0.6b-v2 + shop words (0.5)", "whisper-large-v3-turbo + shop words"]:
    if "files" in res4.get(m, {}):
        before, after = score(m, sets=("halfwheel",)), score(m, fix=snap, sets=("halfwheel",))
        rows[m] = {"entities": before["entities"], "entities, snapped": after["entities"],
                   "WER": before["halfwheel phone"], "WER, snapped": after["halfwheel phone"],
                   "false snaps": false_snaps(m)}
snapped = pd.DataFrame(rows).T
snapped.style.format({c: "{:.1%}" for c in ["entities", "entities, snapped", "WER", "WER, snapped"]} | {"false snaps": "{:.0f}"})

In [ ]:
for new_customer in ["Hannah Smith", "Daniel Okoro", "Raj Patil"]:
    print(f"{new_customer:13} -> {snap(f'Hi, this is {new_customer}, I am a new customer.')}")

## Rerun the models

Everything above used the saved results. The cells below rerun the models that produced them, on a GPU (in Colab:
Runtime, then Change runtime type, then T4 GPU). Each step builds the software it needs in its own environment the
first time, which takes a few minutes, then runs. Set `VOTF_QUICK=2` first for a two-item check that everything
works; leave it unset for a full rerun, which overwrites the saved results. On GPUs without bfloat16, such as the
T4, the scripts use float16 automatically.

In [ ]:
import os
os.environ["VOTF_QUICK"] = "2"  # remove this line for a full rerun

Halfwheel's calls: Kokoro reads them, Chapter 2's phone line carries them.

In [ ]:
!bash tools/stack.sh tts
!~/.votf/envs/tts/bin/python sim/halfwheel_calls.py

80 LibriSpeech recordings (downloads about 700 MB).

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/librispeech_subset.py

Six recognisers on 110 files; about 15 minutes on a T4.

In [ ]:
!bash tools/stack.sh nemo
!~/.votf/envs/nemo/bin/python sim/capture_ch4.py